In [14]:
import pandas as pd
import numpy as np

In [15]:
files={
    'relaxed':'EmoRecData/louis3_7_5min_baseline.csv',
    'stress':'EmoRecData/louis3_7_5min_stress.csv',
    'focus':'EmoRecData/louis3_7_5min_focus.csv',
    'distract':'EmoRecData/louis3_7_5min_distract.csv'
}

columns = [
    "host_time_s",'t_ms',
    "ax1","ay1","az1",'roll1','pitch1','yaw1',"gx1","gy1","gz1","yaw1","pitch1","roll1",
    "ax2","ay2","az2",'roll2','pitch2','yaw2',"gx2","gy2","gz2","yaw2","pitch2","roll2",
    "emg1","emg2"
]


In [16]:
dfs = {}
for emotion, filepath in files.items():
    df = pd.read_csv(filepath)  # just let pandas read the header naturally
    df["label"] = emotion
    dfs[emotion] = df
    print(f"{emotion}: {len(df)} rows")
    print(f"  columns: {df.columns.tolist()}")  # verify they match

cols_to_drop = [
    "host_time_s", "t_ms",
    "roll1","pitch1","yaw1",
    "roll2","pitch2","yaw2"
]

for emotion in dfs:
    dfs[emotion] = dfs[emotion].drop(columns=cols_to_drop)

relaxed: 11545 rows
  columns: ['host_time_s', 't_ms', 'ax1', 'ay1', 'az1', 'roll1', 'pitch1', 'yaw1', 'gx1', 'gy1', 'gz1', 'ax2', 'ay2', 'az2', 'roll2', 'pitch2', 'yaw2', 'gx2', 'gy2', 'gz2', 'emg1', 'emg2', 'label']
stress: 11494 rows
  columns: ['host_time_s', 't_ms', 'ax1', 'ay1', 'az1', 'roll1', 'pitch1', 'yaw1', 'gx1', 'gy1', 'gz1', 'ax2', 'ay2', 'az2', 'roll2', 'pitch2', 'yaw2', 'gx2', 'gy2', 'gz2', 'emg1', 'emg2', 'label']
focus: 11236 rows
  columns: ['host_time_s', 't_ms', 'ax1', 'ay1', 'az1', 'roll1', 'pitch1', 'yaw1', 'gx1', 'gy1', 'gz1', 'ax2', 'ay2', 'az2', 'roll2', 'pitch2', 'yaw2', 'gx2', 'gy2', 'gz2', 'emg1', 'emg2', 'label']
distract: 11390 rows
  columns: ['host_time_s', 't_ms', 'ax1', 'ay1', 'az1', 'roll1', 'pitch1', 'yaw1', 'gx1', 'gy1', 'gz1', 'ax2', 'ay2', 'az2', 'roll2', 'pitch2', 'yaw2', 'gx2', 'gy2', 'gz2', 'emg1', 'emg2', 'label']


In [17]:
df_all = pd.concat(dfs.values(), ignore_index=True)
labels = df_all["label"].to_numpy()
df_all = df_all.drop(columns=["label"])

print("\nRemaining columns:", df_all.columns.tolist())  # verify 14
print("Combined shape:", df_all.shape)                  # [N_rows, 14]



Remaining columns: ['ax1', 'ay1', 'az1', 'gx1', 'gy1', 'gz1', 'ax2', 'ay2', 'az2', 'gx2', 'gy2', 'gz2', 'emg1', 'emg2']
Combined shape: (45665, 14)


In [18]:
data = df_all.to_numpy(dtype=np.float32)

mean = data.mean(axis=0)  # [14]
std  = data.std(axis=0)   # [14]
data_norm = (data - mean) / (std + 1e-8)

np.save("channel_mean.npy", mean)
np.save("channel_std.npy",  std)

In [19]:
WINDOW_SIZE = 125
STEP_SIZE   = 25

all_windows = []
all_labels  = []

for start in range(0, len(data_norm) - WINDOW_SIZE, STEP_SIZE):

    window = data_norm[start : start + WINDOW_SIZE]  # [125, 14]
    all_windows.append(window)

    #window = data_norm[start : start + WINDOW_SIZE]  # [125, 14]
    #window = window.T                                  # [14, 125]
    #all_windows.append(window)
    all_labels.append(labels[start + WINDOW_SIZE // 2])

all_windows = np.array(all_windows, dtype=np.float32)
all_labels  = np.array(all_labels)
print("\nTotal windows:", all_windows.shape)            # [N_windows, 14, 125]



Total windows: (735, 125, 14)


In [21]:
calib_samples = []

for emotion in files.keys():
    indices = np.where(all_labels == emotion)[0]
    chosen  = np.random.choice(indices, 25, replace=False)
    calib_samples.append(all_windows[chosen])
    print(f"{emotion}: {len(indices)} windows available, 25 selected")

calib_data = np.concatenate(calib_samples, axis=0)     # [100, 14, 125]
assert calib_data.shape == (100, 125, 14), f"Wrong shape: {calib_data.shape}"
np.save("calib_data.npy", calib_data)
print("\nSaved calib_data.npy:", calib_data.shape)      # ✅ [100, 14, 125]

relaxed: 186 windows available, 25 selected
stress: 185 windows available, 25 selected
focus: 181 windows available, 25 selected
distract: 183 windows available, 25 selected

Saved calib_data.npy: (100, 125, 14)
